# MOVIE RECOMMENDER SYSTEM

## Project Introduction
This project aims to develop a machine learning model that predicts movie ratings based on user–movie interactions and movie-related features. The goal is to generate accurate rating predictions for unseen user–movie pairs using the provided training and test datasets.
The project implements a hybrid recommendation approach that combines two models: Singular Value Decomposition (SVD), which uses collaborative filtering to learn patterns from historical user ratings, and LightGBM, which predicts ratings using engineered movie features derived from tag-genome relevance scores and available IMDb metadata.
The models are evaluated using Root Mean Squared Error (RMSE), where lower values indicate better prediction accuracy. A validation dataset is used to compare model performance and determine the optimal weight for combining their predictions. Finally, both models are retrained using the full training dataset, and their predictions are combined to generate the final submission.csv file.

## Project Workflow
The project is organized into the following major steps:

1.	Environment Setup and Data Loading

3.	Data Splitting and Baseline Modeling 
4.	Movie Feature Engineering
5.	LightGBM Model Training 
6.	Ensemble Optimization 
7.	Final Model Training 
8.	Prediction and Submission Generation 

## Step 1: Set up the environment and load the data

In [1]:
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from surprise import Dataset, Reader, SVD
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

# Suppress non-critical warnings for cleaner notebook output.
warnings.filterwarnings("ignore")

# Set a fixed random seed for reproducible results.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# Loading data
train = pd.read_csv(
    "train.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32",
        "rating": "float32"
    }
)

test = pd.read_csv(
    "test.csv",
    dtype={
        "userId": "int32",
        "movieId": "int32"
    }
)

In [3]:
print("Training shape:", train.shape)
print("Test shape:", test.shape)

Training shape: (10000038, 4)
Test shape: (5000019, 2)


In [4]:
display(train.head())
display(test.head())

,userId,movieId,rating,timestamp
0,5163,57669,4.0,1518349992
1,106343,5,4.5,1206238739
2,146790,5459,5.0,1076215539
3,106362,32296,2.0,1423042565
4,9041,366,3.0,833375837


,userId,movieId
0,1,2011
1,1,4144
2,1,5767
3,1,6711
4,1,7318


In [5]:
# Check for missing values in the training data.
print("\nMissing values in training data:")
print(train.isnull().sum())


Missing values in training data:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [6]:
# Examine the distribution of known ratings.
print("\nRating statistics:")
print(train["rating"].describe())


Rating statistics:
count    1.000004e+07
mean     3.533395e+00
std      1.036790e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64


## Step 2: Split the data and train the baseline model

In [7]:
# Splitting the data
train_part, val_part = train_test_split(
    train,
    test_size=0.10,
    random_state=RANDOM_STATE
)

# Reset indices for easier handling.
train_part = train_part.reset_index(drop=True)
val_part = val_part.reset_index(drop=True)

print("Training rows:", len(train_part))
print("Validation rows:", len(val_part))

Training rows: 9000034
Validation rows: 1000004


In [9]:
reader = Reader(rating_scale=(0.5, 5.0))

# Candidate configurations to test
configs = [
    (100, 20, 0.005, 0.05),
    (150, 30, 0.005, 0.05),
    (100, 30, 0.005, 0.02),
    (150, 30, 0.003, 0.05),
    (200, 30, 0.005, 0.05),
]

# Prepare the training dataset once
data = Dataset.load_from_df(
    train_part[["userId", "movieId", "rating"]],
    reader
)
full_trainset = data.build_full_trainset()

# Track the best configuration
results = []
best_rmse = float("inf")
best_svd = None
best_params = None
best_predictions = None

# Train and evaluate each configuration
for factors, epochs, lr, reg in configs:
    start = time.time() 

    model = SVD(
        n_factors=factors,
        n_epochs=epochs,
        lr_all=lr,
        reg_all=reg,
        random_state=RANDOM_STATE
    )

    model.fit(full_trainset)

    predictions = np.array([
        model.predict(int(user), int(movie)).est
        for user, movie in zip(
            val_part["userId"],
            val_part["movieId"]
        )
    ])
    rmse = np.sqrt(mean_squared_error(
    val_part["rating"].values,
    predictions
    ))
    results.append({
       "n_factors": factors,
       "n_epochs": epochs,
       "learning_rate": lr,
       "regularization": reg,
       "RMSE": rmse,
       "seconds": round(time.time() - start, 2)
    })

    print(
        f"Factors={factors}, Epochs={epochs}, "
        f"RMSE={rmse:.5f}, "
        f"Time={time.time() - start:.1f}s"
    )
    if rmse < best_rmse:
        best_rmse = rmse
        best_svd = model
        best_predictions = predictions
        best_params = {
            "n_factors": factors,
            "n_epochs": epochs,
            "lr_all": lr,
            "reg_all": reg
        }

# Display results, best first
results_df = pd.DataFrame(results).sort_values("RMSE")

print("\nBest SVD parameters:", best_params)
print(f"Best validation RMSE: {best_rmse:.6f}")

display(results_df)

# Keep these variables for the next notebook steps
svd_model = best_svd
svd_predictions = best_predictions
svd_rmse = best_rmse

Factors=100, Epochs=20, RMSE=0.83852, Time=209.5s
Factors=150, Epochs=30, RMSE=0.82137, Time=286.4s
Factors=100, Epochs=30, RMSE=0.82905, Time=238.6s
Factors=150, Epochs=30, RMSE=0.83931, Time=283.7s
Factors=200, Epochs=30, RMSE=0.82072, Time=866.0s

Best SVD parameters: {'n_factors': 200, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.05}
Best validation RMSE: 0.820716


,n_factors,n_epochs,learning_rate,regularization,RMSE,seconds
4,200,30,0.005,0.05,0.820716,866.04
1,150,30,0.005,0.05,0.821373,286.44
2,100,30,0.005,0.02,0.829047,238.60
0,100,20,0.005,0.05,0.838518,209.51
3,150,30,0.003,0.05,0.839308,283.70


## Step 3: Engineer movie features

In [10]:
# Select informative tag-genome features
chunk_size = 1_000_000

# Running totals allow statistics to be calculated across chunks.
tag_sum = {}
tag_sq_sum = {}
tag_count = {}

for chunk in pd.read_csv(
    "genome_scores.csv",
    usecols=["tagId", "relevance"],
    dtype={
        "tagId": "int32",
        "relevance": "float32"
    },
    chunksize=chunk_size
):
    # Calculate per-tag sums and observation counts.
    grouped = chunk.groupby("tagId")["relevance"].agg(
        ["sum", "count"]
    )
    # Calculate squared relevance totals for variance.
    squared = (
        chunk.assign(
            squared=chunk["relevance"] ** 2
        )
        .groupby("tagId")["squared"]
        .sum()
    )

    # Accumulate statistics across chunks.
    for tag_id, row in grouped.iterrows():
        tag_id = int(tag_id)
        tag_sum[tag_id] = (
            tag_sum.get(tag_id, 0.0)
            + float(row["sum"])
        )
        tag_sq_sum[tag_id] = (
            tag_sq_sum.get(tag_id, 0.0)
            + float(squared.loc[tag_id])
        )
        tag_count[tag_id] = (
            tag_count.get(tag_id, 0)
            + int(row["count"])
        )
# Calculate global mean and variance for each tag.
tag_ids = np.array(list(tag_sum.keys()))

counts = np.array(
    [tag_count[tag_id] for tag_id in tag_ids],
    dtype=np.float64
)
means = np.array([
    tag_sum[tag_id] for tag_id in tag_ids
]) / counts

variances = (
    np.array([
        tag_sq_sum[tag_id] for tag_id in tag_ids
    ]) / counts
    - means ** 2
)
# Rank tags by variance, from highest to lowest.
tag_variance = pd.DataFrame({
    "tagId": tag_ids,
    "variance": variances
}).sort_values(
    "variance",
    ascending=False
)
# Keep the 80 highest-variance tags.
selected_tags = (
    tag_variance.head(80)["tagId"].tolist()
)
print("Selected tags:", len(selected_tags))
print("Top 10 tags by variance:")

display(tag_variance.head(10))

# Release temporary dictionaries to reduce memory use.
del tag_sum, tag_sq_sum, tag_count
gc.collect()

Selected tags: 80
Top 10 tags by variance:


,tagId,variance
229,230,0.080184
18,19,0.063892
845,846,0.058287
521,522,0.057670
1019,1020,0.057518
539,540,0.055903
556,557,0.055075
269,270,0.054883
322,323,0.054406
1083,1084,0.054314


0

In [ ]:
# Load and transform the selected genome features
genome_chunks = []
for chunk in pd.read_csv(
    "genome_scores.csv",
    usecols=["movieId", "tagId", "relevance"],
    dtype={
        "movieId": "int32",
        "tagId": "int32",
        "relevance": "float32"
    },
    chunksize=chunk_size
):
    # Retain only the selected tag IDs.
    filtered = chunk[
        chunk["tagId"].isin(selected_tags)
    ]

    if not filtered.empty:
        genome_chunks.append(filtered)

# Combine filtered chunks.
genome_selected = pd.concat(
    genome_chunks,
    ignore_index=True
)
del genome_chunks
gc.collect()

# Pivot to one row per movie and one column per tag.
genome_features = genome_selected.pivot(
    index="movieId",
    columns="tagId",
    values="relevance"
)

# Give the feature columns descriptive names.
genome_features.columns = [
    f"genome_{tag}"
    for tag in genome_features.columns
]

# Use memory-efficient numeric types and fill missing scores.
genome_features = (
    genome_features
    .astype("float32")
    .fillna(0)
)
genome_features = genome_features.reset_index()
print("Genome feature shape:", genome_features.shape)
display(genome_features.head())

# Release intermediate data.
del genome_selected
gc.collect()

In [ ]:
# Merge IMDb metadata with genome features
links = pd.read_csv("links.csv")
imdb = pd.read_csv(
    "imdb_data.csv",
    low_memory=False
)
print("Links columns:", links.columns.tolist())
print("IMDb columns:", imdb.columns.tolist())
if (
    "imdbId" in links.columns
    and "imdbId" in imdb.columns
):
    # Normalize IMDb identifiers before joining.
    links["imdbId"] = pd.to_numeric(
        links["imdbId"],
        errors="coerce"
    )
    imdb["imdbId"] = pd.to_numeric(
        imdb["imdbId"],
        errors="coerce"
    )
    # Remove records with missing or duplicate IMDb identifiers.
    imdb = imdb.dropna(subset=["imdbId"])
    imdb = imdb.drop_duplicates("imdbId")
    # Connect movie IDs to IMDb metadata.
    imdb_features = links.merge(
        imdb,
        on="imdbId",
        how="left",
        suffixes=("", "_imdb")
    )

    imdb_features = imdb_features.drop_duplicates(
        "movieId"
    )
    # Remove identifiers that should not be model features.
    imdb_features = imdb_features.drop(
        columns=["imdbId", "tmdbId"],
        errors="ignore"
    )
    # Merge genome and IMDb features by movie ID.
    movie_features = genome_features.merge(
        imdb_features,
        on="movieId",
        how="left",
        suffixes=("", "_meta")
    )
else:
    # Fallback if the IMDb join keys are unavailable.
    movie_features = genome_features.copy()
    print(
        "IMDb join keys unavailable; "
        "using genome features."
    )
print("Movie features shape:", movie_features.shape)